<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 8B: *Fire Ignition Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

from rich.console import Console

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_ignition = pd.read_csv('../data/processed/X_ignition.csv')
y_ignition = pd.read_csv('../data/processed/y_ignition.csv').squeeze()  # Load as Series      
details_ignition = pd.read_csv('../data/processed/details_ignition.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/ignition_best_strategy.csv')

with open('../data/processed/model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('../data/processed/feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_ignition,y_ignition], axis=1)
subset = subset_df(reform, 'Target_Ignition', 500)

y = subset['Target_Ignition']
X = subset.drop(columns='Target_Ignition')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
ignition_xgb = xgb.XGBClassifier(**XGB_parameters)
ignition_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 5,
 'min_samples_split': 2,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 2,
 'n_estimators': 150,
 'max_depth': 6,
 'learning_rate': 0.4,
 'verbosity': 0}

In [7]:
models = {
    "XGB": ignition_xgb, 
    "RF": ignition_rf
}

## SHAP

In [8]:
xgb_top = get_shap(ignition_xgb, X_xgb, y_xgb)
rf_top = get_shap(ignition_rf, X_rf, y_rf,)

In [9]:
weather_features = []

drop = ['fire_count','fire_count 3 Day Sum','fire_count 7 Day Sum','fire_count 30 Day Sum',
       'road_density_x_forest_percent','power_line_density_x_log_total_housing']
weather_sets = ['Water Demand','Water Supply','Water Supply Indexes','Fire Danger','Interactions','Wind Slope',
                              'Lag 3 Day Features','Lag 7 Day Features','Lag 30 Day Features']

for weather_set in weather_sets:
    for feature in feature_sets[weather_set]:
        if feature not in drop:
            weather_features.append(feature)

In [10]:
top_xgb_weather = xgb_top.loc[xgb_top['Feature'].isin(weather_features)]
top_xgb_weather[:5]

,Feature,SHAP Importance
4,Wind Speed_x_elevation_range,0.160981
5,Maximum Relative Humidity,0.155165
6,Minimum Relative Humidity 3 Day Mean,0.154636
8,Specific Humidity 30 Day Mean,0.144405
9,SPI 30-Day,0.141727


In [11]:
top_rf_weather = rf_top.loc[rf_top['Feature'].isin(weather_features)]
top_rf_weather[:5]

,Feature,SHAP Importance
20,northness_mean_x_Daily Maximum Air Temperature,0.004167
21,Specific Humidity 30 Day Mean,0.003595
22,Solar Radiation 30 Day Sum,0.003521
23,Wind Speed 7 Day Median_x_northness_mean,0.003479
24,Actual Evapotranspiration 7 Day Sum,0.003474


In [12]:
common_ranked = (
    rf_top[:20][['Feature', 'SHAP Importance']]
    .rename(columns={'SHAP Importance': 'RF_Importance'})
    .merge(
        xgb_top[:20][['Feature', 'SHAP Importance']]
        .rename(columns={'SHAP Importance': 'XGB_Importance'}),
        on='Feature',
        how='inner'
    )
    .sort_values('XGB_Importance', ascending=False)
    .reset_index(drop=True)
)

In [13]:
xgb_top[:10]

,Feature,SHAP Importance
0,fire_count 30 Day Sum,0.861971
1,days_since_last_fire,0.740492
2,dist_to_fires_same_day,0.472951
3,avg_dist_to_fires_same_day,0.164204
4,Wind Speed_x_elevation_range,0.160981
5,Maximum Relative Humidity,0.155165
6,Minimum Relative Humidity 3 Day Mean,0.154636
7,power_line_density_x_log_total_housing,0.150195
8,Specific Humidity 30 Day Mean,0.144405
9,SPI 30-Day,0.141727


In [14]:
rf_top[:10]

,Feature,SHAP Importance
0,fire_count 30 Day Sum,0.037596
1,days_since_last_fire,0.028735
2,dist_to_fires_same_day,0.025897
3,log_total_population,0.016265
4,fire_count 7 Day Sum,0.015696
5,fire_count 3 Day Sum,0.014216
6,log_total_housing,0.013766
7,fire_count,0.013618
8,power_line_density_x_log_total_housing,0.013208
9,intermix_zone,0.012933


In [15]:
common_ranked

,Feature,RF_Importance,XGB_Importance
0,fire_count 30 Day Sum,0.037596,0.861971
1,days_since_last_fire,0.028735,0.740492
2,dist_to_fires_same_day,0.025897,0.472951
3,power_line_density_x_log_total_housing,0.013208,0.150195
4,influence_zone,0.006525,0.105643


## Set Ablation

In [16]:
Ablation_single_column = ablation(models, X, y, feature_sets, best_strategy, single_set = True)

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Infrastructure
Testing RF: Infrastructure
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others
Testing XGB: Coded Ecoregions
Testing RF: Coded Ecoregions
Testing XGB: Coded Seasons
Testing RF: Coded Seasons
Testing XGB: Spatial
Testing RF: Spatial
Testing XGB: Lag 3 Day Features
Testing RF: Lag 3 Day Features
Testing XGB: Lag 7 Day Features
Testing RF: Lag 7 Day Features
Testing XGB: Lag 30 Day Features
Testing RF: Lag 30 Day Features


In [17]:
# Assume your table is in a DataFrame called df
# Filter out the full runs
full_model = Ablation_single_column[Ablation_single_column['Test Set'] == 'Full'][['Model', 'Macro F1']].rename(columns={'Macro F1': 'Full Macro F1'})

# Merge back to the main dataframe on 'Model'
pivot_ablation = Ablation_single_column.merge(full_model, on='Model', how='left')

# Drop columns you don’t need for now
pivot_ablation = pivot_ablation.drop(columns=['Weighted F1', 'High Risk Recall'])

largest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] > 20]
smallest_drops = pivot_ablation[pivot_ablation['Macro F1 Percent Difference'] < 1]

In [18]:
largest_drops

,Test Set,Model,Macro F1,Macro F1 Percent Difference,Full Macro F1
1,Water Demand,XGB,0.595101,27.857009,0.824891
2,Water Supply,XGB,0.579622,29.733508,0.824891
3,Water Supply Indexes,XGB,0.514988,37.568945,0.824891
4,Fire Danger,XGB,0.649965,21.205911,0.824891
13,Others,XGB,0.654991,20.596573,0.824891
15,Coded Seasons,XGB,0.647266,21.533068,0.824891
21,Water Demand,RF,0.636603,22.832313,0.824961
22,Water Supply,RF,0.577798,29.960490,0.824961
23,Water Supply Indexes,RF,0.624540,24.294560,0.824961
32,Wind Slope,RF,0.653953,20.729160,0.824961


In [19]:
smallest_drops

,Test Set,Model,Macro F1,Macro F1 Percent Difference,Full Macro F1
0,Full,XGB,0.824891,0.000000,0.824891
19,Lag 30 Day Features,XGB,0.829847,-0.600844,0.824891
20,Full,RF,0.824961,0.000000,0.824961
36,Spatial,RF,0.829847,-0.592301,0.824961
39,Lag 30 Day Features,RF,0.844903,-2.417382,0.824961
